In [1]:
import torch
import numpy as np
from scipy.stats import pearsonr
from torch_geometric.loader import DataLoader

def evaluate_v3(model, data_list, device, batch_size=64):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for start in range(0, len(data_list), batch_size):
            chunk = data_list[start:start + batch_size]
            loader = DataLoader(chunk, batch_size=len(chunk), shuffle=False)
            batch = next(iter(loader)).to(device)
            pred = model(batch.x, batch.edge_index, batch.batch, batch.target_idx, device)
            preds.append(pred.cpu().numpy())
            targets.append(batch.y.cpu().numpy())
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    errors = np.abs(preds - targets)
    rmse = np.sqrt(np.mean((preds - targets)**2))
    mae = errors.mean()
    corr, _ = pearsonr(preds, targets)

    print(f"  RMSE: {rmse:.4f}, MAE: {mae:.4f}, Pearson r: {corr:.4f}")
    for tol in [0.5, 1.0, 1.5]:
        print(f"  Acc@±{tol}: {(errors <= tol).mean()*100:.1f}%")

    # per-range breakdown (this is what actually tells us if the extremes improved)
    for low, high in [(0,4), (4,4.5), (4.5,7.5), (7.5,8), (8,14)]:
        mask = (targets >= low) & (targets < high)
        n = mask.sum()
        if n == 0:
            continue
        print(f"    pIC50 [{low},{high}): n={n}, MAE={errors[mask].mean():.3f}, Acc@±1.0={(errors[mask]<=1.0).mean()*100:.1f}%")

    return preds, targets

In [2]:
!nvidia-smi

Wed Sep 30 13:02:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 582.08                 Driver Version: 582.08         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 2000 Ada Gene...  WDDM  |   00000000:47:00.0  On |                  Off |
| 30%   54C    P0             17W /   70W |     811MiB /  16380MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import os
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import esm

from torch_geometric.nn import GCNConv, global_mean_pool

DATA_DIR = r"C:\DTI_GNN\data"
CHECKPOINT_DIR = r"C:\DTI_GNN\checkpoints"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load target data
with open(os.path.join(DATA_DIR, "target_sequences.pkl"), "rb") as f:
    target_sequences = pickle.load(f)

# Load ESM-2 35M
print("Loading ESM-2 35M...")
esm_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
batch_converter = alphabet.get_batch_converter()

# Same freezing configuration as training
for p in esm_model.parameters():
    p.requires_grad = False

last_layer_idx = len(esm_model.layers) - 1

for p in esm_model.layers[last_layer_idx].parameters():
    p.requires_grad = True

esm_model = esm_model.to(device)

print("ESM loaded.")

Device: cuda
Loading ESM-2 35M...
ESM loaded.


In [6]:
class DrugEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels=128, out_channels=128):
        super().__init__()

        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = self.conv3(x, edge_index)

        return global_mean_pool(x, batch)


class TargetEncoder(nn.Module):
    def __init__(
        self,
        esm_model,
        batch_converter,
        target_sequences,
        hidden_dim=128,
        esm_dim=480
    ):
        super().__init__()

        self.esm_model = esm_model
        self.batch_converter = batch_converter
        self.target_sequences = target_sequences

        self.proj = nn.Sequential(
            nn.Linear(esm_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )

    def forward(self, target_idx_batch, device):
        unique_idx, inverse = torch.unique(
            target_idx_batch,
            return_inverse=True
        )

        seqs = [
            (str(i.item()), self.target_sequences[i.item()])
            for i in unique_idx
        ]

        _, _, tokens = self.batch_converter(seqs)
        tokens = tokens.to(device)

        results = self.esm_model(
            tokens,
            repr_layers=[12],
            return_contacts=False
        )

        reps = results["representations"][12]

        pooled = torch.stack([
            reps[i, 1:len(seq) + 1].mean(0)
            for i, (_, seq) in enumerate(seqs)
        ])

        per_example = pooled[inverse]

        return self.proj(per_example)


class DTIModelV3(nn.Module):
    def __init__(
        self,
        drug_in_channels,
        esm_model,
        batch_converter,
        target_sequences,
        hidden_dim=128
    ):
        super().__init__()

        self.drug_encoder = DrugEncoder(
            drug_in_channels,
            hidden_dim,
            hidden_dim
        )

        self.target_encoder = TargetEncoder(
            esm_model,
            batch_converter,
            target_sequences,
            hidden_dim
        )

        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(hidden_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(
        self,
        x,
        edge_index,
        batch,
        target_idx,
        device
    ):
        drug_vec = self.drug_encoder(
            x,
            edge_index,
            batch
        )

        target_vec = self.target_encoder(
            target_idx,
            device
        )

        combined = torch.cat(
            [drug_vec, target_vec],
            dim=1
        )

        return self.fusion(combined).squeeze(-1)

In [7]:
model = DTIModelV3(
    drug_in_channels=18,
    esm_model=esm_model,
    batch_converter=batch_converter,
    target_sequences=target_sequences,
    hidden_dim=128
).to(device)

ckpt_path = os.path.join(
    CHECKPOINT_DIR,
    "dti_model_v3_epoch15.pt"
)

ckpt = torch.load(
    ckpt_path,
    weights_only=False
)

model.load_state_dict(
    ckpt["model_state_dict"]
)

model.eval()

print(
    f"Loaded checkpoint from epoch {ckpt['epoch']}, "
    f"train loss {ckpt['train_loss']:.4f}"
)

Loaded checkpoint from epoch 15, train loss 0.2677


In [8]:
import os
DATA_DIR = r"C:\DTI_GNN\data"
CHECKPOINT_DIR = r"C:\DTI_GNN\checkpoints"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# reload final checkpoint into a fresh model instance
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "dti_model_v3_epoch15.pt"), weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded checkpoint from epoch {ckpt['epoch']}, train loss {ckpt['train_loss']:.4f}")

Loaded checkpoint from epoch 15, train loss 0.2677


In [9]:
import numpy as np
from scipy.stats import pearsonr
from torch_geometric.loader import DataLoader

def evaluate_v3(model, data_list, device, batch_size=64, name="Split"):
    model.eval()

    preds = []
    targets = []

    with torch.no_grad():
        for start in range(0, len(data_list), batch_size):

            chunk = data_list[start:start + batch_size]

            loader = DataLoader(
                chunk,
                batch_size=len(chunk),
                shuffle=False
            )

            batch = next(iter(loader)).to(device)

            pred = model(
                batch.x,
                batch.edge_index,
                batch.batch,
                batch.target_idx,
                device
            )

            preds.append(pred.cpu().numpy())
            targets.append(batch.y.cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    errors = np.abs(preds - targets)

    rmse = np.sqrt(np.mean((preds - targets) ** 2))
    mae = errors.mean()

    # Pearson correlation
    if len(np.unique(preds)) > 1 and len(np.unique(targets)) > 1:
        corr, _ = pearsonr(preds, targets)
    else:
        corr = np.nan

    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    print(f"Samples     : {len(targets)}")
    print(f"RMSE        : {rmse:.4f}")
    print(f"MAE         : {mae:.4f}")
    print(f"Pearson r   : {corr:.4f}")

    for tol in [0.5, 1.0, 1.5]:
        acc = (errors <= tol).mean() * 100
        print(f"Acc@±{tol:<3}    : {acc:.2f}%")

    # ---------------------------------------------------------
    # Per-range breakdown
    # ---------------------------------------------------------

    print("\nPer-range breakdown")
    print("-" * 65)

    ranges = [
        (0, 4),
        (4, 4.5),
        (4.5, 7.5),
        (7.5, 8),
        (8, 14)
    ]

    for low, high in ranges:

        mask = (
            (targets >= low) &
            (targets < high)
        )

        n = mask.sum()

        if n == 0:
            continue

        range_mae = errors[mask].mean()
        range_rmse = np.sqrt(
            np.mean((preds[mask] - targets[mask]) ** 2)
        )
        range_acc = (
            (errors[mask] <= 1.0).mean() * 100
        )

        print(
            f"pIC50 [{low:>3},{high:<3}) | "
            f"n={n:<6} | "
            f"MAE={range_mae:.3f} | "
            f"RMSE={range_rmse:.3f} | "
            f"Acc@±1={range_acc:.1f}%"
        )

    return preds, targets

In [10]:
EVAL_DIR = r"C:\DTI_GNN\evaluation"

cold_drug = torch.load(
    os.path.join(EVAL_DIR, "cold_drug_data_v2.pt"),
    weights_only=False
)

cold_pair = torch.load(
    os.path.join(EVAL_DIR, "cold_pair_data_v2.pt"),
    weights_only=False
)

cold_target = torch.load(
    os.path.join(EVAL_DIR, "cold_target_data_v2.pt"),
    weights_only=False
)

print("Cold-drug samples  :", len(cold_drug))
print("Cold-pair samples  :", len(cold_pair))
print("Cold-target samples:", len(cold_target))

Cold-drug samples  : 138556
Cold-pair samples  : 31839
Cold-target samples: 123438


In [ ]:
pred_drug, y_drug = evaluate_v3(
    model, cold_drug, device,
    batch_size=64,
    name="COLD-DRUG"
)

pred_pair, y_pair = evaluate_v3(
    model, cold_pair, device,
    batch_size=64,
    name="COLD-PAIR"
)

pred_target, y_target = evaluate_v3(
    model, cold_target, device,
    batch_size=64,
    name="COLD-TARGET"
)